# Modems and lab analyzers: AT commands and ASTM

Two serial-port veterans that are still everywhere: cellular modules driven with AT commands, and laboratory analyzers that send results with ASTM E1394. Both run against simulators here.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.10.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.10.0-preview.1"

## A cellular module
Commands end with OK or an error; unsolicited result codes (URCs) such as `+CEREG` and `+CMTI` arrive at any time.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.AtCommand;
using IoTCom.Net.Transports;

var modemLink = new InMemoryTransportListener("modem");
await using var module = AtModemSimulator.Create(o => { o.ListenInMemory(modemLink); o.RegistrationDelay = TimeSpan.FromMilliseconds(300); });
await module.StartAsync();
await using var modem = AtModem.Create(o => o.UseInMemory(modemLink));
await modem.ConnectAsync();
await modem.SendCheckedAsync("AT+CEREG=2");
await Task.Delay(600);
var info = await modem.GetInfoAsync();
Console.WriteLine($"{info.Model} · {info.Registration} on {info.Operator} ({info.AccessTechnology}) · {info.Signal}");
Console.WriteLine(await modem.SendAsync("AT+CPIN=\"0000\""));

## SMS in and out

In [ ]:
await modem.SendSmsAsync("+6281234567890", "Pompa air nyala");
var inbox = new TaskCompletionSource<AtUrc>();
modem.UrcReceived += (_, u) => { if (u.Name == "+CMTI") inbox.TrySetResult(u); };
module.DeliverSms("+6289876543210", "STATUS?");
var cmti = await inbox.Task;
var sms = await modem.ReadSmsAsync(int.Parse(cmti.Values[1]));
Console.WriteLine($"{sms.Sender}: {sms.Text}");

## A chemistry analyzer over ASTM
The analyzer sends ENQ, then one framed record at a time, each acknowledged; a corrupted frame is answered with NAK and sent again.

In [ ]:
using IoTCom.Net.Protocols.Astm;

var labLink = new InMemoryTransportListener("lab");
await using var lis = AstmReceiver.Create(o => o.ListenInMemory(labLink));
await lis.StartAsync();
var resultArrived = new TaskCompletionSource<AstmMessage>();
lis.MessageReceived += (_, m) => resultArrived.TrySetResult(m);
await using var analyzer = AstmSender.Create(o => o.UseInMemory(labLink));
await analyzer.ConnectAsync();
analyzer.CorruptNextFrame = true;
await analyzer.SendAsync(new AnalyzerSimulator().NextResult());
var labResult = await resultArrived.Task;
foreach (var r in labResult.Results) Console.WriteLine($"{r.TestCode,-5} {r.Value,7} {r.Units,-7} {r.Flag}");
Console.WriteLine($"retransmitted frames: {analyzer.Retransmissions}");

## Bridge to HL7
Most hospitals want results as HL7 ORU^R01.

In [ ]:
Console.WriteLine(AstmToHl7.ToOru(labResult).Encode().Replace("\r", "\n"));

## Going further
`iotcom at info --sim`, `iotcom astm listen --hl7` and the AstmAnalyzerBridge sample. Synthetic data only; not a medical device. See `docs/en/protocols/at-commands.md` and `docs/en/protocols/astm.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*